# Clustering dengan Recursive Spherical K-Means

## Install Spherical K-Means

### Library ini ternyata udah Obsolete dengan versi Python sekarang

In [23]:
# pip install spherecluster

## Import Library

In [24]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
import os
import csv

print("Library berhasil dimuat.")

Library berhasil dimuat.


## Load Data Vektor

In [25]:
input_file = '../dataset/tickets_normalized.npy'
X_norm = np.load(input_file)

print(f"Vektor ternormalisasi berhasil dimuat. Dimensi: {X_norm.shape}")

Vektor ternormalisasi berhasil dimuat. Dimensi: (1639, 256)


## Clustering & Evaluasi Silhouette Score

### Level 1

#### Membuat klaster level 1 dan melakukan evaluasi nilai silhoette score

In [58]:
# Jumlah klaster level 1 (2-10)
k = 10


print(f"=== EVALUASI LEVEL 1 dengan k = {k} ===")

# Menjalankan Spherical K-Means Level 1
skm_l1 = KMeans(n_clusters=k, n_init=10, random_state=42)
labels_l1 = skm_l1.fit_predict(X_norm)

# Mengambil pusat klaster dan memaksa panjangnya menjadi 1 (Unit Vector)
centroids_l1 = normalize(skm_l1.cluster_centers_, norm='l2')

# Evaluasi Silhouette Score Level 1
score_l1 = silhouette_score(X_norm, labels_l1)

print(f"=== HASIL LEVEL 1 dengan k = {k} ===")
print(f"Silhouette Score (Level 1): {score_l1:.4f}")
print("Distribusi data per klaster:")
unique, counts = np.unique(labels_l1, return_counts=True)
print(dict(zip(unique, counts)))

=== EVALUASI LEVEL 1 dengan k = 10 ===
=== HASIL LEVEL 1 dengan k = 10 ===
Silhouette Score (Level 1): 0.0605
Distribusi data per klaster:
{np.int32(0): np.int64(218), np.int32(1): np.int64(353), np.int32(2): np.int64(142), np.int32(3): np.int64(145), np.int32(4): np.int64(185), np.int32(5): np.int64(50), np.int32(6): np.int64(137), np.int32(7): np.int64(244), np.int32(8): np.int64(120), np.int32(9): np.int64(45)}


In [59]:
def save_silhouette_log(filename, score):
    """
    Menyimpan nama file dan nilai silhouette score ke dalam satu file log.
    Melakukan append jika file sudah ada.
    """
    log_path = "../hasil/log_silhouette_scores.csv"
    
    # Cek apakah folder sudah ada
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    
    # Tentukan apakah perlu menulis header (jika file baru)
    file_exists = os.path.isfile(log_path)
    
    with open(log_path, mode='a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['filename', 'silhouette_score']) # Header
        
        writer.writerow([filename, f"{score:.4f}"])

### Save Klaster Level 1

#### Menyimpan hasil klaster 1 dengan menggabungkan dengan preprocessed data

In [60]:
output_dir = '../hasil/'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Direktori {output_dir} berhasil dibuat.")

df_preprocessed = pd.read_csv('../dataset/tickets_preprocessed.csv')

df_labels_l1 = pd.DataFrame({
    'KLASTER': labels_l1
})

df_l1_final = pd.concat([df_preprocessed, df_labels_l1], axis=1)

filename_l1_merged = f"{output_dir}Hasil_Cluster_L1_K{k}.csv"
df_l1_final.to_csv(filename_l1_merged, index=False)

# Simpan log silhouette score
save_silhouette_log(filename_l1_merged, score_l1)

print(f"Data preprocessed dan label Level 1 (k={k}) berhasil digabungkan.")
print(f"File tersimpan: {filename_l1_merged}")
print(df_l1_final[['CLEAN_TEXT', 'KLASTER']].head())

Data preprocessed dan label Level 1 (k=10) berhasil digabungkan.
File tersimpan: ../hasil/Hasil_Cluster_L1_K10.csv
                                          CLEAN_TEXT  KLASTER
0  setwapres medsos crawlback comment siang tim i...        6
1  selamat pagi tim it mohon bantuannya saya mene...        1
2  bps data postingan tidak masuk selamat malam t...        1
3  selamat sore mas dan tim info untuk siputri ti...        4
4  heinz dashboard loading selamat sore tim it mo...        2


### Level 2

#### Membuat klaster level 2 untuk setiap sub-klaster, melakukan evaluasi silhouette score dan menyimpan file hasil klaster

In [61]:
# Jumlah klaster untuk level 2 (2-10)
# k_sub = 10


# Inisialisasi
final_hierarchy = np.empty(len(X_norm), dtype=object)
unique_clusters = np.unique(labels_l1)

for k_sub in range(2, 11):
    print(f"=== EVALUASI LEVEL 2 PER ITERASI SUB-KLASTER (k = {k} dan k_sub = {k_sub}) ===")
    for cluster_id in unique_clusters:
        # Ambil index dan subset data berdasarkan klaster Level 1
        idx = np.where(labels_l1 == cluster_id)[0]
        X_subset = X_norm[idx]
        
        # Syarat minimal data untuk clustering (minimal sebanyak k_sub)
        if len(idx) >= k_sub:
            skm_l2 = KMeans(n_clusters=k_sub, random_state=42)
            labels_l2 = skm_l2.fit_predict(X_subset)

            # IMPLEMENTASI TEORI SKM: Normalisasi Centroid Sub-Klaster
            centroids_l2 = normalize(skm_l2.cluster_centers_, norm='l2')
            
            # Hitung Silhouette Score UNTUK ITERASI INI SAJA
            s_l2_iteration = silhouette_score(X_subset, labels_l2)
            print(f"Klaster Level 1 dengan k = {k}, Sub-Klaster ke-{cluster_id} -> Hasil Silhouette Score Sub-Klaster: {s_l2_iteration:.4f}")
            
            # Simpan hasil klastering level 2 untuk sub-klaster ini
            filename_sub = f"../hasil/Label_L1_K{k}_L2_K{k_sub}_{cluster_id}.csv"
            
            df_sub = pd.DataFrame({
                'original_index': idx,
                'label_l1': cluster_id,
                'label_l2': labels_l2
            })
            df_sub.to_csv(filename_sub, index=False)
            print(f"   -> File disimpan: {filename_sub}")

            # Simpan log silhouette score untuk sub-klaster ini
            save_silhouette_log(filename_sub, s_l2_iteration)

            # Simpan label hirarki (Parent-Child)
            for i, sub_label in enumerate(labels_l2):
                final_hierarchy[idx[i]] = f"{cluster_id}-{sub_label}"
        else:
            # Jika data tidak cukup untuk di-split sesuai k_sub
            print(f"Klaster Level 1 dengan k = {k}, Sub-Klaster ke-{cluster_id} -> Data tidak cukup ({len(idx)} data) untuk di-split menjadi {k_sub}")
            for i in range(len(idx)):
                final_hierarchy[idx[i]] = f"{cluster_id}-static"

    print("-" * 50)
    print(f"=== EVALUASI LEVEL 2 PER ITERASI SUB-KLASTER (k = {k} dan k_sub = {k_sub}) SELESAI ===")

=== EVALUASI LEVEL 2 PER ITERASI SUB-KLASTER (k = 10 dan k_sub = 2) ===
Klaster Level 1 dengan k = 10, Sub-Klaster ke-0 -> Hasil Silhouette Score Sub-Klaster: 0.0298
   -> File disimpan: ../hasil/Label_L1_K10_L2_K2_0.csv
Klaster Level 1 dengan k = 10, Sub-Klaster ke-1 -> Hasil Silhouette Score Sub-Klaster: 0.0508
   -> File disimpan: ../hasil/Label_L1_K10_L2_K2_1.csv
Klaster Level 1 dengan k = 10, Sub-Klaster ke-2 -> Hasil Silhouette Score Sub-Klaster: 0.0535
   -> File disimpan: ../hasil/Label_L1_K10_L2_K2_2.csv
Klaster Level 1 dengan k = 10, Sub-Klaster ke-3 -> Hasil Silhouette Score Sub-Klaster: 0.0605
   -> File disimpan: ../hasil/Label_L1_K10_L2_K2_3.csv
Klaster Level 1 dengan k = 10, Sub-Klaster ke-4 -> Hasil Silhouette Score Sub-Klaster: 0.0908
   -> File disimpan: ../hasil/Label_L1_K10_L2_K2_4.csv
Klaster Level 1 dengan k = 10, Sub-Klaster ke-5 -> Hasil Silhouette Score Sub-Klaster: 0.2547
   -> File disimpan: ../hasil/Label_L1_K10_L2_K2_5.csv
Klaster Level 1 dengan k = 10, Sub